In [17]:
import re, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, BertTokenizerFast

# Text Encoder

## 參數設定

In [18]:
PILOT_CSV = Path("/home/yingtzudai/Multimodal_SPoD_Video/data/split/pilot_posts.csv")
META_DIR  = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer")
OUT_DIR   = Path("/home/yingtzudai/Multimodal_SPoD_Video/features/text")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ckiplab/bert-base-chinese"   # 對照組可改 "xlm-roberta-base"
MAX_LEN    = 512
BATCH_SIZE = 32
APPLY_MASK = True                            # 跑 ablation 時改 False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## 讀 pilot 並以 media_id 找回貼文文字

In [19]:
# media_id 一律讀成字串：IG 的 id 很長，讀成數字會因浮點精度失真而對不上
pilot = pd.read_csv(PILOT_CSV, dtype={"media_id": str, "id": str})
pilot["media_id"] = pilot["media_id"].str.strip()

def load_texts(pilot: pd.DataFrame) -> pd.Series:
    lookup, missing_files = {}, []
    for user in pilot["username"].unique():
        f = META_DIR / f"{user}-FullVideoInfo.csv"
        if not f.exists():
            missing_files.append(user); continue
        meta = pd.read_csv(f, dtype={"media_id": str})
        meta["media_id"] = meta["media_id"].str.strip()
        meta = meta.drop_duplicates("media_id")
        lookup.update(zip(meta["media_id"], meta["text"]))
    if missing_files:
        warnings.warn(f"找不到 metadata 檔：{missing_files}")
    return pilot["media_id"].map(lookup)

pilot["text_raw"] = load_texts(pilot)
n_missing = pilot["text_raw"].isna().sum()
print(f"共 {len(pilot)} 篇，找不到文字 {n_missing} 篇")
pilot["text_raw"] = pilot["text_raw"].fillna("").astype(str)


共 299 篇，找不到文字 0 篇


## 業配詞刪除 (待確認)

In [ ]:
# Tier 1：明確揭露詞 → 遮
ZH_STRICT = ["業配", "工商", "廣告", "贊助", "置入", "付費合作", "品牌合作",
             "合作邀約", "邀約", "體驗邀請", "公關品", "公關邀約", "折扣碼", "優惠碼","乾爹", "廠商"]
EN_HASHTAG = ["ad", "ads", "advert", "advertisement", "sponsored", "sponsor",
              "sponsorship", "paidpartnership", "partner", "partnership",
              "collab", "gifted", "pr", "promo"]
EN_PHRASE  = [r"paid\s+partnership", r"sponsored\s+by", r"in\s+partnership\s+with",
              r"thanks\s+to\s+\S+\s+for\s+sponsoring"]
OTHER_LANG = ["협찬", "광고", "提供", "タイアップ"]          # 偶爾夾雜的韓、日文
# Tier 2：模糊詞 → 預設不遮，ablation 時可加入
ZH_OPTIONAL = ["合作", "感謝", "團購", "買"]

def build_patterns(zh_terms):
    zh = "|".join(map(re.escape, sorted(zh_terms, key=len, reverse=True)))
    return [
        re.compile(rf"#[^\s#]*(?:{zh})[^\s#]*"),                        # 含揭露詞的整個 hashtag
        re.compile(rf"#(?:{'|'.join(EN_HASHTAG)})\b", re.IGNORECASE),   # #ad 不會誤砍 #adidas
        re.compile(rf"\b(?:{'|'.join(EN_PHRASE)})\b", re.IGNORECASE),
        re.compile(rf"(?:{zh})"),                                       # 剩下的裸字
    ]

PATTERNS = build_patterns(ZH_STRICT + OTHER_LANG)

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)        # 全形＃→#、全形英數→半形
    text = re.sub(r"https?://\S+", " ", text)         # 網址沒有語意，順便省 token
    return re.sub(r"\s+", " ", text).strip()

def mask_sponsor_terms(text: str):
    hits = 0
    for p in PATTERNS:
        text, n = p.subn(" ", text)
        hits += n
    return re.sub(r"\s+", " ", text).strip(), hits

pilot["text_norm"] = pilot["text_raw"].map(normalize)
masked = pilot["text_norm"].map(mask_sponsor_terms)
pilot["text_masked"] = masked.str[0]
pilot["mask_hits"]   = masked.str[1]

# 洩漏檢查：各 label 中被遮到字的貼文比例
print(pilot.assign(has_hit=pilot.mask_hits > 0)
           .groupby("label")["has_hit"].agg(["mean", "sum", "count"]))

# 隨機抽幾篇看前後差異
for _, r in pilot[pilot.mask_hits > 0].sample(min(5, (pilot.mask_hits > 0).sum()),
                                              random_state=0).iterrows():
    print("BEFORE:", r.text_norm[:150]); print("AFTER :", r.text_masked[:150]); print("-" * 60)

           mean  sum  count
label                      
0      0.006135    1    163
1      0.102941   14    136
BEFORE: 雜耍演員變世界第一健美先生💪|志祺七七 #shorts #promo 👊 medisana 56合1|智能八點傳感體脂計 體脂、BMI、心律、肌肉量、趨勢圖⋯ 等 56 項身體數據輕鬆掌握! ❗️連結點不到?從首頁點擊我們的 Portaly頁面也可以進入唷! ✔︎ 5.25吋彩色大螢幕|色彩鮮明清晰
AFTER : 雜耍演員變世界第一健美先生💪|志祺七七 #shorts 👊 medisana 56合1|智能八點傳感體脂計 體脂、BMI、心律、肌肉量、趨勢圖⋯ 等 56 項身體數據輕鬆掌握! ❗️連結點不到?從首頁點擊我們的 Portaly頁面也可以進入唷! ✔︎ 5.25吋彩色大螢幕|色彩鮮明清晰,數據一目瞭然
------------------------------------------------------------
BEFORE: 「今天我們要來開箱,泰國曼谷全新飯店!」 - 前兩年來曼谷玩的時候,一直就看到日本有名的西鐵集團要進駐啦!終於在2024年9月開幕,位於超級熱鬧的silom區,出門不用3分鐘就可以走到地鐵站,超級方便! - 說實話入住前超級緊張,上網刷到滿滿負評,有人說水壓不夠、有人說半夜很吵、有人說水管有臭味,但
AFTER : 「今天我們要來開箱,泰國曼谷全新飯店!」 - 前兩年來曼谷玩的時候,一直就看到日本有名的西鐵集團要進駐啦!終於在2024年9月開幕,位於超級熱鬧的silom區,出門不用3分鐘就可以走到地鐵站,超級方便! - 說實話入住前超級緊張,上網刷到滿滿負評,有人說水壓不夠、有人說半夜很吵、有人說水管有臭味,但
------------------------------------------------------------
BEFORE: 我送我老公的聖誕禮物...你們猜他最後有收嗎😳? 六萬塊的電腦捨得花~當然眼睛也要認真顧! 懷孕、備孕、全家大小都能吃的昕顏靈光凍再次分享給你們🧡 超多人反饋吃一週就有感~真的不是開玩笑😎 微米分子更好吸收,這就是為什麼吃了真的有效! 現在昕顏

## 載入模型

In [21]:
if MODEL_NAME.startswith("ckiplab/"):
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")  # CKIP 官方指定
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

texts = (pilot["text_masked"] if APPLY_MASK else pilot["text_norm"]).tolist()

# 看有多少貼文會被截斷
lens = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in texts]
print(f"token 長度 中位數={np.median(lens):.0f}, P95={np.percentile(lens,95):.0f}, "
      f"超過 {MAX_LEN} 的比例={np.mean(np.array(lens) > MAX_LEN):.1%}")
print("[UNK] 比例：",
      np.mean([tokenizer(t)["input_ids"].count(tokenizer.unk_token_id) / max(len(tokenizer(t)["input_ids"]), 1)
               for t in texts[:200]]).round(4))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Token indices seque

token 長度 中位數=52, P95=433, 超過 512 的比例=2.7%
[UNK] 比例： 0.0508


## 抽 CLS embedding

In [22]:
@torch.no_grad()
def encode_cls(texts, batch_size=BATCH_SIZE):
    out = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                          max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            h = model(**batch).last_hidden_state          # (B, L, 768)
        out.append(h[:, 0].float().cpu())                 # 取位置 0 = [CLS]，不用 pooler_output
    return torch.cat(out)

X_T = encode_cls(texts)
print(X_T.shape)   # (N, 768)

  0%|          | 0/10 [00:00<?, ?it/s]

torch.Size([299, 768])


## 存檔（以 media_id 對齊之後的 video embedding）

In [23]:
tag = f"{MODEL_NAME.split('/')[-1]}_{'masked' if APPLY_MASK else 'raw'}"
torch.save({"media_id": pilot["media_id"].tolist(),
            "label": pilot["label"].tolist(),
            "X_T": X_T,
            "model": MODEL_NAME, "masked": APPLY_MASK, "max_len": MAX_LEN},
           OUT_DIR / f"pilot_text_{tag}.pt")
pilot[["media_id", "username", "label", "mask_hits", "text_masked"]].to_csv(
    OUT_DIR / f"pilot_text_{tag}.csv", index=False)
print("saved:", OUT_DIR / f"pilot_text_{tag}.pt")

# %% Cell 7 ── Sanity check：linear probe（依網紅分組做 CV，避免同一網紅跨 fold）
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

y, groups = pilot["label"].values, pilot["username"].values
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
auc = cross_val_score(LogisticRegression(max_iter=2000), X_T.numpy(), y,
                      groups=groups, cv=cv, scoring="roc_auc")
print(f"[{tag}] linear probe ROC-AUC = {auc.mean():.3f} ± {auc.std():.3f}")
# 解讀：masked 和 raw 各跑一次。raw 的 AUC 如果明顯高出很多（例如接近 1.0），
# 代表揭露詞確實在洩漏標籤，論文裡也可以拿這組對照來說明遮蔽的必要性。

saved: /home/yingtzudai/Multimodal_SPoD_Video/features/text/pilot_text_bert-base-chinese_masked.pt
[bert-base-chinese_masked] linear probe ROC-AUC = 0.787 ± 0.104


## (之後端到端訓練用）可 fine-tune 的 Text Encoder module

In [24]:
import torch.nn as nn

class TextEncoder(nn.Module):
    """F_T：輸出 [CLS] 表示 X^T。freeze=True 對應 pilot 的特徵抽取設定。"""
    def __init__(self, name=MODEL_NAME, freeze=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False
        self.out_dim = self.bert.config.hidden_size

    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids,
                         attention_mask=attention_mask).last_hidden_state[:, 0]